# UrbanEye — Notebook 03: Model Evaluation & Visual Segmentation Benchmarking

**Objective:** Load trained PyTorch U-Net checkpoint (`model_weights.pth`), evaluate performance on test satellite tiles, compute quantitative segmentation metrics (Mean IoU, F1 Score, Precision, Recall, Pixel Accuracy), and save visual segmentation comparison figures to `figures/`.

In [ ]:
import sys
import os
import torch

# Add src to path
sys.path.append(os.path.abspath('../src'))
from data_utils import create_dataloaders
from model_utils import load_model
from eval_utils import calculate_metrics, plot_segmentation_predictions

In [ ]:
# 1. Load test dataset & trained U-Net checkpoint
img_dir = '../sample_data/images'
mask_dir = '../sample_data/masks'
_, _, test_loader = create_dataloaders(img_dir, mask_dir, batch_size=4)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
weights_path = '../model_weights.pth'
if os.path.exists(weights_path):
    model = load_model(weights_path, device=device)
else:
    from model_utils import UNet
    model = UNet().to(device)
    print("Warning: Trained weights not found; evaluated initial architecture.")

In [ ]:
# 2. Perform Inference & Quantitative Evaluation on Test Set
model.eval()
all_images = []
all_masks = []
all_preds = []

with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device), masks.to(device)
        preds = model(images)
        all_images.append(images)
        all_masks.append(masks)
        all_preds.append(preds)

all_images = torch.cat(all_images, dim=0)
all_masks = torch.cat(all_masks, dim=0)
all_preds = torch.cat(all_preds, dim=0)

metrics = calculate_metrics(all_masks, all_preds)

print("=== Quantitative Segmentation Evaluation Results ===")
print(f"Mean IoU (Jaccard Index): {metrics['mean_iou']:.4f}")
print(f"F1 Score (Dice Score)  : {metrics['f1_score']:.4f}")
print(f"Precision              : {metrics['precision']:.4f}")
print(f"Recall                 : {metrics['recall']:.4f}")
print(f"Pixel Accuracy         : {metrics['pixel_accuracy']:.4f}")

In [ ]:
# 3. Plot & Save Visual Segmentation Comparisons
plot_segmentation_predictions(all_images, all_masks, all_preds, num_samples=3, save_path='../figures/segmentation_results.png')